# PySpark Technical Interview Questions - SQL Style Problems

## 📚 Overview

This notebook contains **classic SQL interview problems adapted for PySpark**. These questions are extremely common across all companies (FAANG, fintech, startups).

**Topics Covered:**
- Salary comparisons (manager vs employee)
- N-th highest problems
- Department rankings
- Consecutive pattern detection
- Anti-joins (finding missing data)
- Date-based calculations
- Complex aggregations

**Interview Frequency:** 95%+ of data engineering interviews include at least one SQL-style problem.

---

## 💡 Study Tips

1. **Learn the SQL first** - understand the problem in SQL
2. **Translate to PySpark** - practice the conversion
3. **Focus on window functions** - 80% of problems use them
4. **Draw it out** - sketch data transformations

---

## Setup: Create Spark Session

In [ ]:
from pyspark.sql import SparkSession  # Main entry point for DataFrame API
from pyspark.sql.functions import (  # Built-in functions
    col, lit, when,  # Column operations
    count, sum as _sum, avg, max as _max, min as _min,  # Aggregations
    row_number, rank, dense_rank, lag, lead,  # Window ranking functions
    datediff, to_date, date_add, current_date,  # Date operations
    desc, asc  # Sorting
)
from pyspark.sql.window import Window  # Window specifications

# Create Spark Session
spark = SparkSession.builder \  # Builder for configuration
    .appName("Interview_SQL_Problems") \  # App name (visible in Spark UI)
    .master("local[*]") \  # Local mode using all CPU cores
    .config("spark.sql.shuffle.partitions", "4") \  # Shuffle partitions (4 for local, scale up for production)
    .getOrCreate()  # Create new or get existing session

spark.sparkContext.setLogLevel("ERROR")  # Only show errors

print("✅ Spark Session Created")  # Confirmation
print(f"Spark Version: {spark.version}")  # Display version

---

# 🔀 SHUFFLE PARTITIONS IN SQL-STYLE PROBLEMS

## Why It Matters for These Questions

Many SQL-style interview problems involve operations that **trigger shuffles**:

### Operations That Shuffle:

| Question Type | Operation | Shuffle Trigger |
|---------------|-----------|----------------|
| **Self-joins** | `emp.join(mgr, ...)` | Data redistributed by join key |
| **Rankings** | `dense_rank().over(Window.partitionBy(...))` | Data partitioned by group |
| **Aggregations** | `groupBy("dept").agg(sum(...))` | Data grouped by key |
| **Deduplication** | `groupBy("email").count()` | Data grouped by email |
| **Sorting** | `orderBy("salary")` | Global sort requires shuffle |

## 📊 Our Configuration:

```python
.config("spark.sql.shuffle.partitions", "4")
```

**Why 4 partitions?**
- Small sample data in interview problems
- Running locally on laptop
- Default 200 would be overkill

**For real data, calculate:**
```
Partitions = Data Size MB / 128 MB
```

## 🎯 Example: Top N Per Group

```python
# This triggers a shuffle by 'department'
window = Window.partitionBy("department").orderBy(col("salary").desc())
df.withColumn("rank", dense_rank().over(window))
```

**What happens:**
1. Data shuffled into 4 partitions by department
2. Each partition contains all rows for certain departments
3. Ranking computed independently per partition
4. Results combined

## 💡 Interview Tips:

### When Asked About Performance:

**Interviewer:** "How would this perform on 1 TB of data?"

**You:** "I'd increase shuffle partitions to ~8,000 (1TB / 128MB). I'd also check for data skew - if some departments are much larger, I might use salting or enable AQE for automatic skew handling."

### When Explaining Your Solution:

✅ **Good:** "This groupBy will trigger a shuffle to redistribute data by email address. With our 4 partitions, it's fine for small data, but we'd scale this up for production."

❌ **Bad:** "This groups the data." (Too vague)

---

📖 **Detailed guide**: [shuffle_partitions_explanation.md](shuffle_partitions_explanation.md)

---

---

# QUESTION 1: Employees Earning More Than Their Managers

## 📝 Problem Statement (LeetCode #181)

Given an `Employee` table with columns `id`, `name`, `salary`, and `manager_id`, write a query to find employees who earn more than their managers.

```sql
-- SQL Solution
SELECT e1.name AS Employee
FROM Employee e1
JOIN Employee e2 ON e1.manager_id = e2.id
WHERE e1.salary > e2.salary;
```

## 🎯 Interview Focus
- **Frequency**: Very common (80%+ of interviews)
- **Tests**: Self-joins, understanding relationships
- **Follow-up**: "What if multiple levels of managers?"

## 💡 Key Concepts
- **Self-join**: Joining table to itself
- **Aliasing**: Using different names for same table
- **Null handling**: Managers have NULL manager_id

In [ ]:
# Create Employee table
# manager_id references id of another employee (self-reference)
data = [
    (1, "John", 100000, None),      # John is CEO (no manager)
    (2, "Jane", 90000, 1),          # Jane reports to John
    (3, "Bob", 95000, 1),           # Bob reports to John (earns LESS than John)
    (4, "Alice", 110000, 1),        # Alice reports to John (earns MORE - answer!)
    (5, "Charlie", 92000, 2),       # Charlie reports to Jane
    (6, "David", 93000, 2),         # David reports to Jane (earns MORE - answer!)
]

employees = spark.createDataFrame(data, ["id", "name", "salary", "manager_id"])

print("Employee Table:")
employees.show()

print("\n💡 Notice:")
print("  - John (CEO) has no manager (NULL manager_id)")
print("  - Alice earns $110k, more than John's $100k")
print("  - David earns $93k, more than Jane's $90k")

In [ ]:
# SOLUTION: Self-join to compare employee with manager

# Step 1: Alias the same table twice (employees as 'emp' and 'mgr')
# Step 2: Join on emp.manager_id = mgr.id
# Step 3: Filter where emp.salary > mgr.salary

emp = employees.alias("emp")     # Employee perspective
mgr = employees.alias("mgr")     # Manager perspective

# Join employees with their managers
result = emp.join(
    mgr,
    col("emp.manager_id") == col("mgr.id"),  # Join condition
    "inner"  # Inner join excludes CEOs (no manager)
).where(
    col("emp.salary") > col("mgr.salary")  # Filter condition
).select(
    col("emp.name").alias("Employee"),
    col("emp.salary").alias("Employee_Salary"),
    col("mgr.name").alias("Manager"),
    col("mgr.salary").alias("Manager_Salary")
)

print("\n📊 Employees Earning More Than Their Managers:")
result.show()

# Interview Tip: Explain that this is a "self-join" and why aliasing is needed

### ✅ Key Takeaways - Question 1

1. **Self-join pattern**: Join table to itself with different aliases
2. **Join condition**: Link child to parent via foreign key
3. **Null handling**: CEO/top level has NULL manager_id
4. **Performance**: Consider using broadcast if manager table is small

**Interview Follow-ups:**
- Q: "Find employees earning 2x their manager?" → Change condition to `emp.salary > mgr.salary * 2`
- Q: "Show all employees and their manager's salary?" → Use LEFT join instead of INNER
- Q: "Find employees with no manager?" → `employees.filter(col("manager_id").isNull())`

---

# QUESTION 2: Department Top 3 Salaries

## 📝 Problem Statement (LeetCode #185)

Find the top 3 highest salaries in each department.

```sql
-- SQL Solution
SELECT Department, Name, Salary
FROM (
  SELECT *, 
    DENSE_RANK() OVER (PARTITION BY Department ORDER BY Salary DESC) as rnk
  FROM Employee
) WHERE rnk <= 3;
```

## 🎯 Interview Focus
- **Frequency**: EXTREMELY common (90%+ of interviews)
- **Tests**: Window functions, ranking, partitioning
- **Follow-up**: "Why dense_rank instead of rank?"

## 💡 Key Concepts
- **Window function**: Operate over groups (partitions)
- **PARTITION BY**: Creates separate rankings per department
- **dense_rank()**: Handles ties without gaps (1,1,2 not 1,1,3)
- **Top N per group**: Most common window function pattern

In [ ]:
# Create employee data with departments
data = [
    ("Engineering", "Alice", 95000),
    ("Engineering", "Bob", 90000),
    ("Engineering", "Charlie", 90000),  # Tied for 2nd
    ("Engineering", "David", 85000),
    ("Engineering", "Eve", 80000),
    ("Sales", "Frank", 75000),
    ("Sales", "Grace", 70000),
    ("Sales", "Henry", 70000),  # Tied for 2nd
    ("Sales", "Ivy", 65000),
    ("HR", "Jack", 60000),
    ("HR", "Karen", 55000),
]

employees = spark.createDataFrame(data, ["department", "name", "salary"])

print("Employee Salaries by Department:")
employees.orderBy("department", col("salary").desc()).show(15)

print("\n💡 Notice the tied salaries:")
print("  - Engineering: Charlie & Bob both at $90k")
print("  - Sales: Grace & Henry both at $70k")

In [ ]:
# SOLUTION: Top 3 salaries per department using dense_rank

# Step 1: Define window specification
#   - PARTITION BY department: Separate ranking for each dept
#   - ORDER BY salary DESC: Highest salary gets rank 1
window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

# Step 2: Add dense_rank column
# Step 3: Filter for top 3 (rank <= 3)
# Step 4: Sort for presentation

result = employees \
    .withColumn("salary_rank", dense_rank().over(window_spec)) \
    .filter(col("salary_rank") <= 3) \
    .orderBy("department", "salary_rank")

print("\n🏆 Top 3 Salaries Per Department:")
result.show()

print("\n💡 Why dense_rank?")
print("  - Ties get same rank: Bob & Charlie both rank 2")
print("  - Next rank is 3, not 4 (no gaps)")
print("  - Ensures we get exactly top 3 salary levels")

In [ ]:
# COMPARISON: rank() vs row_number() vs dense_rank()

print("\n📊 Ranking Function Comparison:")
print("=" * 60)

window_spec = Window.partitionBy("department").orderBy(col("salary").desc())

comparison = employees \
    .withColumn("row_number", row_number().over(window_spec)) \
    .withColumn("rank", rank().over(window_spec)) \
    .withColumn("dense_rank", dense_rank().over(window_spec)) \
    .filter(col("department") == "Engineering") \
    .orderBy("dense_rank", "name")

print("\nEngineering Department (showing all 3 ranking types):")
comparison.show()

print("\n🔍 Analysis for tied $90k salaries (Bob & Charlie):")
print("  row_number:  2, 3   (breaks ties arbitrarily)")
print("  rank:        2, 2   (ties allowed, skips to 4)")
print("  dense_rank:  2, 2   (ties allowed, continues to 3)")
print("\n✅ For 'top N' problems, use dense_rank!")

### ✅ Key Takeaways - Question 2

1. **Top N per group**: `PARTITION BY + dense_rank() + filter`
2. **Window spec**: `partitionBy(group_col).orderBy(value_col.desc())`
3. **dense_rank() for ties**: Ensures no gaps in rankings
4. **Performance**: Window functions cause shuffle by partition key

**Interview Answers:**
- Q: "Why dense_rank?" → A: "Handles ties without gaps, ensures exactly top N levels"
- Q: "What if no ties?" → A: "All three functions give same result"
- Q: "How to get bottom 3?" → A: "Use orderBy(col.asc()) instead of desc"

---

# QUESTION 3: Consecutive Numbers

## 📝 Problem Statement (LeetCode #180)

Find all numbers that appear at least **three times consecutively** in a table.

```sql
-- SQL Solution using LAG
SELECT DISTINCT num AS ConsecutiveNums
FROM (
  SELECT num,
    LAG(num, 1) OVER (ORDER BY id) AS prev1,
    LAG(num, 2) OVER (ORDER BY id) AS prev2
  FROM Logs
) WHERE num = prev1 AND num = prev2;
```

## 🎯 Interview Focus
- **Frequency**: Common (60%+ of interviews)
- **Tests**: LAG/LEAD, pattern detection, sequences
- **Follow-up**: "Find N consecutive occurrences"

## 💡 Key Concepts
- **LAG function**: Access previous row values
- **Window ordering**: Critical for sequence detection
- **Pattern matching**: Compare current with previous rows

In [ ]:
# Create logs table with consecutive numbers
data = [
    (1, 1),
    (2, 1),
    (3, 1),   # 1 appears 3 times consecutively (rows 1-3)
    (4, 2),
    (5, 1),
    (6, 2),
    (7, 2),   # 2 appears only 2 times (not enough)
    (8, 3),
    (9, 3),
    (10, 3),  # 3 appears 3 times consecutively (rows 8-10)
    (11, 3),  # 3 appears 4 times total!
]

logs = spark.createDataFrame(data, ["id", "num"])

print("Logs Table:")
logs.show()

print("\n💡 Expected Answer: 1 and 3")
print("  - 1 appears in rows 1,2,3 (3 consecutive)")
print("  - 3 appears in rows 8,9,10,11 (4 consecutive!)")
print("  - 2 only appears twice consecutively (not enough)")

In [ ]:
# SOLUTION: Find 3 consecutive occurrences using LAG

# Step 1: Create window ordered by id (chronological order)
window_spec = Window.orderBy("id")

# Step 2: Use LAG to get previous 2 values
#   - lag(col, 1): Value from 1 row back
#   - lag(col, 2): Value from 2 rows back

# Step 3: Check if current = prev1 = prev2 (all three match)

result = logs \
    .withColumn("prev1", lag("num", 1).over(window_spec)) \
    .withColumn("prev2", lag("num", 2).over(window_spec)) \
    .filter(
        (col("num") == col("prev1")) & 
        (col("num") == col("prev2"))
    ) \
    .select("num") \
    .distinct()

print("\n🔢 Numbers Appearing 3+ Times Consecutively:")
result.show()

# Interview Tip: Explain that LAG looks backward, LEAD looks forward

In [ ]:
# VISUALIZATION: Show how LAG works

print("\n📊 Understanding LAG Function:")
print("=" * 60)

window_spec = Window.orderBy("id")

visualization = logs \
    .withColumn("prev1", lag("num", 1).over(window_spec)) \
    .withColumn("prev2", lag("num", 2).over(window_spec)) \
    .withColumn(
        "is_consecutive_3",
        when(
            (col("num") == col("prev1")) & (col("num") == col("prev2")),
            lit("✓ YES")
        ).otherwise(lit("✗ NO"))
    )

print("\nRow-by-row Analysis:")
visualization.show(15, truncate=False)

print("\n💡 How it works:")
print("  - Row 3: num=1, prev1=1, prev2=1 → All match! ✓")
print("  - Row 10: num=3, prev1=3, prev2=3 → All match! ✓")
print("  - Row 7: num=2, prev1=2, prev2=1 → No match ✗")

### ✅ Key Takeaways - Question 3

1. **Pattern**: `LAG(col, n)` to access previous row values
2. **For N consecutive**: Use LAG with offsets 1 through N-1
3. **Window ordering**: MUST order by sequence column (id, timestamp)
4. **Alternative**: Can use LEAD to look forward instead of backward

**Interview Follow-ups:**
- Q: "Find 5 consecutive?" → A: "Use lag(num, 1), lag(num, 2), lag(num, 3), lag(num, 4)"
- Q: "LAG vs LEAD?" → A: "LAG looks back, LEAD looks forward - both work!"
- Q: "What if partition by user?" → A: "Add partitionBy to window spec"

---

# Continue practicing...

The remaining SQL-style questions follow the same pattern:
1. LeetCode-style problem statement
2. SQL solution for reference
3. Step-by-step PySpark solution
4. Detailed explanations

**Practice Questions 4-10:**
- Q4: Customers Who Never Order (Anti-join)
- Q5: Trip Cancellation Rate
- Q6: Rising Temperature (Date comparisons)
- Q7: Exchange Seats (Even/Odd logic)
- Q8: First Login Date
- Q9: Active Users (5+ Consecutive Days)
- Q10: Find Duplicate Emails

💡 **Interview Tip**: Practice explaining your thought process out loud. In interviews, communication is as important as code!